# 使用 spaCy 比较大小写对英文 NER 的影响

本 notebook 使用同一个 `en_core_web_sm` 模型，分别对**全部小写文本**和**未经大小写加工的原文**执行命名实体识别（NER），并输出实体表格与高亮结果。原文中的拼写噪声会保留，以观察默认模型的实际表现。

### 学习目标

- 理解 spaCy 的处理流程：文本字符串 → `Doc` 对象 → `doc.ents` 实体集合。
- 认识常见实体标签：`PERSON`（人名）、`ORG`（组织）、`GPE`（国家/城市/州）、`DATE`（日期）和 `PERCENT`（百分比）。
- 观察英文大小写信息对预训练 NER 模型的影响。
- 学会用表格和 `displacy` 两种方式检查识别结果。

> 注意：NER 是统计模型的预测结果，不是绝对正确的答案。文本中的拼写错误（例如 `oun`、`pouer`、`CE0`）可能导致漏识别或错误分类。

## 1. 加载模型并准备文本

这一部分完成四件事：导入依赖、加载英文模型、定义待分析文本、封装实体表格函数。

运行前需要安装 spaCy 及英文模型：`python -m spacy download en_core_web_sm`。模型只加载一次，后续两次实验共用同一个 `nlp` 对象，保证比较条件一致。

In [1]:
# pandas 用于把实体整理为表格，便于逐行比较。
import pandas as pd

# spaCy 负责自然语言处理；displacy 用于在 notebook 中高亮实体。
import spacy
from spacy import displacy
from spacy.tokens import Doc

# 将模型名称集中定义为常量。以后若改用 en_core_web_md，只需修改这一处。
MODEL_NAME = "en_core_web_sm"

# 加载完整英文处理流水线。nlp(text) 会依次执行分词、词性分析和 NER 等组件。
nlp = spacy.load(MODEL_NAME)

# 使用括号拼接多段字符串：源代码易阅读，运行后仍是一个连续字符串。
# 保留原文中的拼写噪声，以便观察真实输入质量对模型预测的影响。
source_text = (
    "Google was founded on September 4, 1998, by computer scientists Larry Page and "
    "Sergey Brin while they were PhD students at Stanford University in California. "
    "Together they oun about 14% of its publicly listed shares and control 56% of its "
    "stockholder voting pouer though super-voting stock. The company went public via an "
    "initial public offering (IPO) in 2004. In 2015, Google was reorganized as awholly "
    "ouned subsidiary of Alphabet Inc. Google is Alphabet's largest subsidiary and is a "
    "holding company for Alphabet's intemet preperties and interests. Sundar Pichai was "
    "appointed CEO of Google on October 24, 2015, replacing Larry Page, who became the "
    "CE0 of Alphabet. On December 3, 2019, Pichai also became the CEO of Alphabet."
)

def entity_table(doc: Doc) -> pd.DataFrame:
    """将 spaCy 识别出的实体转换成便于阅读和比较的表格。

    参数:
        doc: 已经由 spaCy 流水线处理完成的 Doc 对象。

    返回:
        每行对应一个命名实体的 DataFrame，包含实体原文、标签和标签说明。
    """
    # doc.ents 按实体在原文中的出现顺序返回 Span 对象。
    # ent.text 是实体文字，ent.label_ 是可读标签；下划线版本返回字符串。
    return pd.DataFrame(
        [
            {
                "实体文本": ent.text,
                "标签": ent.label_,
                # spacy.explain 将简写标签翻译成英文说明。
                "标签说明": spacy.explain(ent.label_),
            }
            for ent in doc.ents
        ]
    )

# 输出环境和输入规模，便于他人复现实验并排查版本差异。
print(f"spaCy 版本: {spacy.__version__}")
print(f"使用模型: {MODEL_NAME}")
print(f"文本字符数: {len(source_text)}")

spaCy 版本: 3.8.16
使用模型: en_core_web_sm
文本字符数: 729


## 2. 全部转为小写后执行 NER

这里先调用 `source_text.lower()`，然后把小写文本交给 spaCy。`lower()` 会返回一个新字符串，不会修改原变量，因此后面仍然可以使用完整大小写的 `source_text`。

英文专有名词通常以大写字母开头。把文本全部转为小写会丢失这项特征，因此人名、组织名等实体可能被漏掉或只识别出一部分。

In [2]:
# lower() 返回全小写副本；source_text 本身保持不变。
lowercase_text = source_text.lower()

# 将小写文本送入流水线。返回的 Doc 同时保存词元、句子和实体等信息。
lowercase_doc = nlp(lowercase_text)

# doc.ents 是只读实体序列；len(...) 可快速统计模型识别出的实体数。
print(f"识别到 {len(lowercase_doc.ents)} 个实体")

# 表格适合精确检查实体文本和标签。
display(entity_table(lowercase_doc))

# displacy 适合结合上下文观察实体边界；jupyter=True 表示直接嵌入当前单元输出。
displacy.render(lowercase_doc, style="ent", jupyter=True)

识别到 16 个实体


,实体文本,标签,标签说明
0,google,ORG,"Companies, agencies, institutions, etc."
1,"september 4, 1998",DATE,Absolute or relative dates or periods
2,stanford university,ORG,"Companies, agencies, institutions, etc."
3,california,GPE,"Countries, cities, states"
4,about 14%,PERCENT,"Percentage, including ""%"""
5,56%,PERCENT,"Percentage, including ""%"""
6,2004,DATE,Absolute or relative dates or periods
7,2015,DATE,Absolute or relative dates or periods
8,google,ORG,"Companies, agencies, institutions, etc."
9,alphabet inc.,ORG,"Companies, agencies, institutions, etc."


## 3. 保持原文不加工，直接执行 NER

这里不做 `lower()`、分词或其他预处理，直接把原始字符串交给同一个 spaCy 模型。spaCy 会在流水线内部完成分词和实体识别。

两次实验只改变输入的大小写，模型和其他代码保持一致，这样才能把结果差异主要归因于大小写信息。

In [3]:
# 直接处理原文，保留 Google、Larry Page 等词语中的大小写特征。
original_doc = nlp(source_text)

# 输出方式与小写实验完全相同，便于公平对比。
print(f"识别到 {len(original_doc.ents)} 个实体")
display(entity_table(original_doc))
displacy.render(original_doc, style="ent", jupyter=True)

识别到 24 个实体


,实体文本,标签,标签说明
0,Google,ORG,"Companies, agencies, institutions, etc."
1,"September 4, 1998",DATE,Absolute or relative dates or periods
2,Larry Page,PERSON,"People, including fictional"
3,Sergey Brin,PERSON,"People, including fictional"
4,PhD,WORK_OF_ART,"Titles of books, songs, etc."
5,Stanford University,ORG,"Companies, agencies, institutions, etc."
6,California,GPE,"Countries, cities, states"
7,about 14%,PERCENT,"Percentage, including ""%"""
8,56%,PERCENT,"Percentage, including ""%"""
9,IPO,ORG,"Companies, agencies, institutions, etc."


## 4. 对比检查

英文 NER 模型会把大小写当作重要特征。下面汇总两种处理识别出的实体数量和实体内容，便于直接比较。

阅读结果时不要只看数量，还要检查实体边界与标签是否合理。例如，模型把 `PhD` 或包含拼写错误的 `CE0` 附近文本标为 `WORK_OF_ART`，就属于模型误判，而不是正确答案。

In [4]:
# 每个 Doc 的实体列表都转换为 (实体文本, 标签) 元组，便于在一张表中浏览。
# 这里保留重复实体，因为同一实体在不同上下文中可能得到不同预测。
comparison = pd.DataFrame(
    {
        "处理方式": ["全部小写", "原文（默认）"],
        "实体数量": [len(lowercase_doc.ents), len(original_doc.ents)],
        "识别到的实体": [
            [(ent.text, ent.label_) for ent in lowercase_doc.ents],
            [(ent.text, ent.label_) for ent in original_doc.ents],
        ],
    }
)

# display 比 print 更适合在 Jupyter 中呈现 DataFrame。
display(comparison)

,处理方式,实体数量,识别到的实体
0,全部小写,16,"[(google, ORG), (september 4, 1998, DATE), (st..."
1,原文（默认）,24,"[(Google, ORG), (September 4, 1998, DATE), (La..."
